In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat

In [ ]:
PATH_OR = os.path.join('..', 'content', 'mnist_original.mat')
PATH_AD = os.path.join('..', 'content', 'adversarial_data_1000.mat')

In [3]:
TRAIN_RATIO = 0.9
CLUSTERS = 856
EPOCHS = 10

In [4]:
mnist = loadmat(PATH_OR)
adverserial = loadmat(PATH_AD)

In [5]:
mnist_data = mnist['data'].T
mnist_labels = mnist['label'].reshape(-1)

adversarial_data = adverserial['adversarial_examples']
adversarial_labels = adverserial['labels'].reshape(-1)

In [6]:
if mnist_data.dtype != adversarial_data.dtype:
    mnist_data = mnist_data.astype(adversarial_data.dtype)

combined_data = np.concatenate((mnist_data, adversarial_data), axis=0)
combined_labels = np.concatenate((mnist_labels, adversarial_labels), axis=0)

In [7]:
train_mnist_data, test_mnist_data = mnist_data[:int(len(mnist_data) * TRAIN_RATIO)], mnist_data[int(len(mnist_data) * TRAIN_RATIO):]
train_mnist_labels, test_mnist_labels = mnist_labels[:int(len(mnist_labels) * TRAIN_RATIO)], mnist_labels[int(len(mnist_labels) * TRAIN_RATIO):]

train_adversarial_data, test_adversarial_data = adversarial_data[:int(len(adversarial_data) * TRAIN_RATIO)], adversarial_data[int(len(adversarial_data) * TRAIN_RATIO):]
train_adversarial_labels, test_adversarial_labels = adversarial_labels[:int(len(adversarial_labels) * TRAIN_RATIO)], adversarial_labels[int(len(adversarial_labels) * TRAIN_RATIO):]

train_combined_data, test_combined_data = np.concatenate((train_mnist_data, train_adversarial_data), axis=0), np.concatenate((test_mnist_data, test_adversarial_data), axis=0)
train_combined_labels, test_combined_labels = np.concatenate((train_mnist_labels, train_adversarial_labels), axis=0), np.concatenate((test_mnist_labels, test_adversarial_labels), axis=0)

In [8]:
from sklearn.cluster import KMeans
from scipy.stats import mode

# Normal Training

In [9]:
kmeans_or = KMeans(n_clusters=CLUSTERS, n_init="auto", max_iter=EPOCHS, random_state=42)
kmeans_or.fit(train_mnist_data)
cluster_assignments = kmeans_or.labels_

In [10]:
cluster_labels_or = np.zeros(CLUSTERS, dtype=int)
for i in range(CLUSTERS):
    mask = cluster_assignments == i
    cluster_labels_or[i] = mode(train_mnist_labels[mask], keepdims=True).mode[0]

In [11]:
pred_test_mnist_label_or = cluster_labels_or[kmeans_or.predict(test_mnist_data)]
accuracy = np.mean(pred_test_mnist_label_or == test_mnist_labels)
print(f"Original Accuracy: {accuracy * 100:.2f}%")

pred_test_adverserial_label_or = cluster_labels_or[kmeans_or.predict(adversarial_data)]
accuracy = np.mean(pred_test_adverserial_label_or == adversarial_labels)
print(f"Adverserial Accuracy: {accuracy * 100:.2f}%")

pred_test_combined_label_or = cluster_labels_or[kmeans_or.predict(test_combined_data)]
accuracy = np.mean(pred_test_combined_label_or == test_combined_labels)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")

Original Accuracy: 92.37%
Adverserial Accuracy: 9.70%
Overall Accuracy: 91.20%


# Adverserial Training

In [12]:
kmeans_ad = KMeans(n_clusters=CLUSTERS, n_init="auto", max_iter=EPOCHS, random_state=42)
kmeans_ad.fit(train_combined_data)
cluster_assignments_ad = kmeans_ad.labels_

In [13]:
cluster_labels_ad = np.full(CLUSTERS, -1, dtype=int)
for i in range(CLUSTERS):
    mask = cluster_assignments_ad == i
    if not np.any(mask):
        continue
    cluster_labels_ad[i] = mode(train_combined_labels[mask], keepdims=True).mode[0]

In [14]:
pred_test_mnist_label_ad = cluster_labels_ad[kmeans_ad.predict(test_mnist_data)]
accuracy = np.mean(pred_test_mnist_label_ad == test_mnist_labels)
print(f"Original Accuracy: {accuracy * 100:.2f}%")

pred_test_adverserial_label_ad = cluster_labels_ad[kmeans_ad.predict(test_adversarial_data)]
accuracy = np.mean(pred_test_adverserial_label_ad == test_adversarial_labels)
print(f"Adverserial Accuracy: {accuracy * 100:.2f}%")

pred_test_combined_label_ad = cluster_labels_ad[kmeans_ad.predict(test_combined_data)]
accuracy = np.mean(pred_test_combined_label_ad == test_combined_labels)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")

Original Accuracy: 92.51%
Adverserial Accuracy: 10.00%
Overall Accuracy: 91.35%
